In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/raw/train.csv')

display(df.head())
display(df.info())
display(df.describe())

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
target = 'RiskScore'
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df[target], kde=True, ax=axes[0])
sns.boxplot(x=df[target], ax=axes[1])

### Замечание
- Уже в начале обнаружены аномальные и пустые значения RiskScore. Для дальнейшего исследования - удаляем.
- Также удялаем дубликаты

In [ ]:
print(df.shape)
df = df.dropna(subset=target)
df = df[(df[target] >= 0) & (df[target] <= 100)]
df.drop_duplicates(inplace=True)
print(df.shape)
df.duplicated().sum()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df[target], kde=True, ax=axes[0])
sns.boxplot(x=df[target], ax=axes[1])

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.drop('ApplicationDate')
num_cols = df.select_dtypes(include='number').columns

In [ ]:
import math
# Рассчет строк и столбцов для cat_cols
cat_ncols = 3
cat_nrows = math.ceil(len(cat_cols) / cat_ncols)

# Рассчет строк и столбцов для num_cols
num_ncols = 3
num_nrows = math.ceil(len(num_cols) / num_ncols)


In [ ]:
fig, axes = plt.subplots(cat_nrows, cat_ncols, figsize=(12, 8))
axes = axes.flatten()

for i, col, in enumerate(cat_cols):
    sns.violinplot(data=df, x=target, y=col, ax=axes[i])

for j in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(num_nrows, num_ncols, figsize=(15, 60))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.regplot(data=df, x=target, y=col, ax=axes[i])

for j in range(len(num_cols), len(axes)):
    plt.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
df_corr = df[num_cols].corr(method='pearson')
mask = abs(df_corr) < 0.6

plt.figure(figsize=(15, 10))
sns.heatmap(data=df_corr, annot=True, mask=mask, linewidths=0.5)

In [ ]:
df_corr = df[num_cols].corr(method='spearman')
mask = abs(df_corr) < 0.6

plt.figure(figsize=(15, 10))
sns.heatmap(data=df_corr, annot=True, mask=mask, linewidths=0.5)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

features_for_vif = [col for col in num_cols if col != 'RiskScore']
X = df[features_for_vif].replace([np.inf, -np.inf], np.nan).dropna()
X_const = add_constant(X)

vif_data = pd.DataFrame()
vif_data['Feature'] = X.columns

vif_data['VIF'] = [
    variance_inflation_factor(X_const.values, i)
    for i in range(1, X_const.shape[1])
]

vif_data = vif_data.sort_values(by='VIF', ascending=False).reset_index(drop=True)
vif_data

In [ ]:
import missingno as msno
msno.matrix(df)


In [ ]:
df['ApplicationDate'] = pd.to_datetime(df['ApplicationDate'])
df_date = df.copy()
df_date['Year'] = df['ApplicationDate'].dt.year
df_date['Month'] = df['ApplicationDate'].dt.month
df_date['DayOfWeek'] = df['ApplicationDate'].dt.dayofweek
df_date['Day'] = df['ApplicationDate'].dt.day

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()
sns.lineplot(data=df_date, y=target, x='Year', ax=axes[0])
sns.lineplot(data=df_date, y=target, x='Month', ax=axes[1])
sns.lineplot(data=df_date, y=target, x='DayOfWeek', ax=axes[2])
sns.lineplot(data=df_date, y=target, x='Day', ax=axes[3])

### Выводы

- Бимодальная выборка с четким разделением по RiskScore около 50. Подходит для модели классификации.
- Сезонность: Пониженный риск: февраль и июль - отпускники | 7-10 и 25-26 числа месяца - выдача зарплаты | выходные (добавить бинарные признаки для этих дат is_vacation, is_salary, is_weekend).
- Сильная корреляция около 1 (колонки-дублёры: AnnualIncome, Experience, LoanAmount, TotalAssets, BaseInterestRate) - удалить один из признаков, остальные - преобразовать логически.
- MonthlyIncome и CreditScore - нелинейные зависимости. Первое - логарифмируем(для регрессии/для леса оставляем как есть), второе - оставляем как есть (четкое разделение по кредитному рейтингу подойдет для задачи классификации).
- Пропуски: заполнение медианной по train выборке.